# Phase 3 Final ETL & Pipeline Pack

## Objective
This notebook focuses on learning ETL pipeline thinking using PySpark.

The goal is to move from isolated queries to a structured workflow:
**Extract → Transform → Load**

This phase focuses on:
- data ingestion
- cleaning
- transformation
- pipeline building

## Core ETL Concept

Every data engineering workflow follows ETL:
- **Extract** = read data
- **Transform** = clean, filter, join, aggregate
- **Load** = save or display final output


### Step 1: Start Spark Session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Phase3_ETL") \
    .getOrCreate()

### Step 2: Upload Dataset

In Databricks
- Catalog → Upload Files

Suppose Databricks gives:

/Volumes/workspace/default/files/customers.csv

Use that path.

### Step 3: Read Customer Dataset

In [0]:
customers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/customers.csv")
display(customers)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
2,Emma,Jones,emma.jones@webmail.com,555-0002,456 Oak St,Centerville,OH,45459
3,Olivia,Brown,olivia.brown@outlook.com,555-0003,789 Pine St,Greenville,SC,29601
4,Liam,Johnson,liam.johnson@gmail.com,555-0004,101 Maple St,Riverside,CA,92501
5,Noah,Williams,noah.williams@yahoo.com,555-0005,202 Birch St,Lakeside,TX,75001
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
7,Isabella,Davis,isabella.davis@icloud.com,555-0007,404 Spruce St,Boise,ID,83701
8,James,Martinez,james.martinez@live.com,555-0008,505 Walnut St,Des Moines,IA,50301
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


### Step 4: Inspect Data

In [0]:
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: integer (nullable = true)



In [0]:
# rows
customers.show(5)

+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
|customer_id|first_name|last_name|               email|phone_number|     address|       city|state|zip_code|
+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
|          1|      John|    Smith|john.smith@domain...|    555-0001|  123 Elm St|Springfield|   IL|   62701|
|          2|      Emma|    Jones|emma.jones@webmai...|    555-0002|  456 Oak St|Centerville|   OH|   45459|
|          3|    Olivia|    Brown|olivia.brown@outl...|    555-0003| 789 Pine St| Greenville|   SC|   29601|
|          4|      Liam|  Johnson|liam.johnson@gmai...|    555-0004|101 Maple St|  Riverside|   CA|   92501|
|          5|      Noah| Williams|noah.williams@yah...|    555-0005|202 Birch St|   Lakeside|   TX|   75001|
+-----------+----------+---------+--------------------+------------+------------+-----------+-----+--------+
only showing top 5 

In [0]:
#columns: 
customers.columns

['customer_id',
 'first_name',
 'last_name',
 'email',
 'phone_number',
 'address',
 'city',
 'state',
 'zip_code']

In [0]:
#count:
customers.count()

50

### Step 5: Check Missing Values

In [0]:
from pyspark.sql.functions import col, sum

customers.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers.columns
]).show()

+-----------+----------+---------+-----+------------+-------+----+-----+--------+
|customer_id|first_name|last_name|email|phone_number|address|city|state|zip_code|
+-----------+----------+---------+-----+------------+-------+----+-----+--------+
|          0|         0|        0|    0|           0|      0|   0|    0|       0|
+-----------+----------+---------+-----+------------+-------+----+-----+--------+



### Step 6: Remove Missing Values

In [0]:
customers_clean = customers.dropna()

In [0]:
print("Original:", customers.count())
print("After dropna:", customers_clean.count())

Original: 50
After dropna: 50


### Step 7: Remove Duplicate Records
Duplicates can affect reports, so remove them.

In [0]:
customers_clean = customers_clean.dropDuplicates()

print("Original Count :", customers.count())
print("After Cleaning :", customers_clean.count())

Original Count : 50
After Cleaning : 50


In [0]:
#Display the cleaned data:
display(customers_clean)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
31,Chloe,Adams,chloe.adams@aol.com,555-0031,2828 Elm St,San Jose,CA,95101
33,Grace,Baker,grace.baker@live.com,555-0033,3030 Cedar St,Jackson,MS,39201
40,Benjamin,Evans,benjamin.evans@zoho.com,555-0040,3737 Elm St,Denver,CO,80202
43,Lily,Turner,lily.turner@webmail.com,555-0043,4040 Spruce St,Chicago,IL,60601
49,Elena,Gray,elena.gray@zoho.com,555-0049,4646 Elm St,Detroit,MI,48202
44,Daniel,Morris,daniel.morris@zoho.com,555-0044,4141 Walnut St,San Diego,CA,92102
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


### Step 8: Filter Records

In [0]:
from pyspark.sql.functions import col

customers_filtered = customers_clean.filter(
    (col("customer_id").isNotNull()) &
    (col("city").isNotNull())
)

display(customers_filtered)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
31,Chloe,Adams,chloe.adams@aol.com,555-0031,2828 Elm St,San Jose,CA,95101
33,Grace,Baker,grace.baker@live.com,555-0033,3030 Cedar St,Jackson,MS,39201
40,Benjamin,Evans,benjamin.evans@zoho.com,555-0040,3737 Elm St,Denver,CO,80202
43,Lily,Turner,lily.turner@webmail.com,555-0043,4040 Spruce St,Chicago,IL,60601
49,Elena,Gray,elena.gray@zoho.com,555-0049,4646 Elm St,Detroit,MI,48202
44,Daniel,Morris,daniel.morris@zoho.com,555-0044,4141 Walnut St,San Diego,CA,92102
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


### Step 9: Read JSON and Parquet Files

In [0]:
customers.write.mode("overwrite").json("/Volumes/workspace/default/databricks2027/customers_json")
json_df = spark.read.json("/Volumes/workspace/default/databricks2027/customers_json")

display(json_df)

address,city,customer_id,email,first_name,last_name,phone_number,state,zip_code
123 Elm St,Springfield,1,john.smith@domain.com,John,Smith,555-0001,IL,62701
456 Oak St,Centerville,2,emma.jones@webmail.com,Emma,Jones,555-0002,OH,45459
789 Pine St,Greenville,3,olivia.brown@outlook.com,Olivia,Brown,555-0003,SC,29601
101 Maple St,Riverside,4,liam.johnson@gmail.com,Liam,Johnson,555-0004,CA,92501
202 Birch St,Lakeside,5,noah.williams@yahoo.com,Noah,Williams,555-0005,TX,75001
303 Cedar St,Oakland,6,alice.miller@aol.com,Alice,Miller,555-0006,CA,94601
404 Spruce St,Boise,7,isabella.davis@icloud.com,Isabella,Davis,555-0007,ID,83701
505 Walnut St,Des Moines,8,james.martinez@live.com,James,Martinez,555-0008,IA,50301
606 Cherry St,Albany,9,sophia.garcia@zoho.com,Sophia,Garcia,555-0009,NY,12201
707 Maple St,Portland,10,lucas.rodriguez@hotmail.com,Lucas,Rodriguez,555-0010,OR,97201


### Step 10: Create a Parquet file

In [0]:
customers.write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/databricks2027/customers_parquet")
parquet_df = spark.read.parquet(
    "/Volumes/workspace/default/databricks2027/customers_parquet"
)
display(parquet_df)

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,John,Smith,john.smith@domain.com,555-0001,123 Elm St,Springfield,IL,62701
2,Emma,Jones,emma.jones@webmail.com,555-0002,456 Oak St,Centerville,OH,45459
3,Olivia,Brown,olivia.brown@outlook.com,555-0003,789 Pine St,Greenville,SC,29601
4,Liam,Johnson,liam.johnson@gmail.com,555-0004,101 Maple St,Riverside,CA,92501
5,Noah,Williams,noah.williams@yahoo.com,555-0005,202 Birch St,Lakeside,TX,75001
6,Alice,Miller,alice.miller@aol.com,555-0006,303 Cedar St,Oakland,CA,94601
7,Isabella,Davis,isabella.davis@icloud.com,555-0007,404 Spruce St,Boise,ID,83701
8,James,Martinez,james.martinez@live.com,555-0008,505 Walnut St,Des Moines,IA,50301
9,Sophia,Garcia,sophia.garcia@zoho.com,555-0009,606 Cherry St,Albany,NY,12201
10,Lucas,Rodriguez,lucas.rodriguez@hotmail.com,555-0010,707 Maple St,Portland,OR,97201


### Business Pipeline Exercise 1: Read Sales Data → Clean Nulls → Calculate Daily Sales

#### Step 1: Read sales.csv

In [0]:
sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/databricks2027/sales.csv")

display(sales)

sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,1,1,2024-01-15,2,39.98
2,1,3,2024-01-20,1,29.99
3,2,2,2024-01-16,1,25.0
4,2,4,2024-01-22,3,89.97
5,3,5,2024-01-17,2,49.98
6,4,6,2024-01-18,4,119.96
7,4,7,2024-01-25,1,15.5
8,5,8,2024-01-19,3,66.75
9,6,9,2024-01-20,2,40.0
10,7,10,2024-01-21,5,110.95


#### Step 2: Check Schema

In [0]:
sales.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



#### Step 3: Check Missing Values

In [0]:
from pyspark.sql.functions import col, sum

sales.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in sales.columns
]).show()

+-------+-----------+----------+---------+--------+------------+
|sale_id|customer_id|product_id|sale_date|quantity|total_amount|
+-------+-----------+----------+---------+--------+------------+
|      0|          0|         0|        0|       0|           0|
+-------+-----------+----------+---------+--------+------------+



#### Step 4: Remove Null Values

In [0]:
sales_clean = sales.dropna()

display(sales_clean)

sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,1,1,2024-01-15,2,39.98
2,1,3,2024-01-20,1,29.99
3,2,2,2024-01-16,1,25.0
4,2,4,2024-01-22,3,89.97
5,3,5,2024-01-17,2,49.98
6,4,6,2024-01-18,4,119.96
7,4,7,2024-01-25,1,15.5
8,5,8,2024-01-19,3,66.75
9,6,9,2024-01-20,2,40.0
10,7,10,2024-01-21,5,110.95


#### Create SQL View

In [0]:
sales_clean.createOrReplaceTempView("sales")

#### Step 5: Calculate Daily Sales

In [0]:
sales.columns

['sale_id',
 'customer_id',
 'product_id',
 'sale_date',
 'quantity',
 'total_amount']

- Daily Sales (SQL)

In [0]:
%sql
SELECT
    sale_date,
    SUM(total_amount) AS daily_sales
FROM sales
GROUP BY sale_date;

sale_date,daily_sales
2024-01-20,69.99
2024-02-22,60.0
2024-01-17,49.98
2024-01-21,110.95
2024-01-26,67.47
2024-02-14,79.96
2024-03-03,55.5
2024-02-07,89.97
2024-01-29,92.0
2024-02-01,19.99


- Daily Sales (PySpark)

In [0]:
from pyspark.sql.functions import sum

daily_sales = sales_clean.groupBy("sale_date") \
    .agg(sum("total_amount").alias("daily_sales"))

display(daily_sales)

sale_date,daily_sales
2024-01-20,69.99
2024-02-22,60.0
2024-01-17,49.98
2024-01-21,110.95
2024-01-26,67.47
2024-02-14,79.96
2024-03-03,55.5
2024-02-07,89.97
2024-01-29,92.0
2024-02-01,19.99


### Business Pipeline Exercise 2: City-wise Revenue

In [0]:
customers_clean.createOrReplaceTempView("customers")
sales_clean.createOrReplaceTempView("sales")

- City-wise Revenue(SQL)

In [0]:
%sql
SELECT
    c.city,
    SUM(s.total_amount) AS city_revenue
FROM customers c
JOIN sales s
ON c.customer_id = s.customer_id
GROUP BY c.city
ORDER BY city_revenue DESC;

city,city_revenue
Riverside,135.45999999999998
Boston,119.96
Centerville,114.97
Las Vegas,112.47
Boise,110.95
New York,109.96
Seattle,104.99
St. Louis,99.95
San Diego,95.5
Washington,92.0


City-wise Revenue(PySpark)

In [0]:
from pyspark.sql.functions import sum

customer_sales = customers_clean.join(
    sales_clean,
    on="customer_id",
    how="inner"
)
city_revenue = customer_sales.groupBy("city") \
    .agg(sum("total_amount").alias("city_revenue")) \
    .orderBy("city_revenue", ascending=False)

display(city_revenue)

city,city_revenue
Riverside,135.45999999999998
Boston,119.96
Centerville,114.97
Las Vegas,112.47
Boise,110.95
New York,109.96
Seattle,104.99
St. Louis,99.95
San Diego,95.5
Washington,92.0


### Business Pipeline Exercise 3: Find Repeat Customers (>2 Orders)

In [0]:
sales_clean.createOrReplaceTempView("sales_clean")

- SQL

In [0]:
%sql
SELECT customer_id,
       COUNT(sale_id) AS order_count
FROM sales_clean
GROUP BY customer_id
HAVING COUNT(sale_id) > 2
ORDER BY order_count DESC;

customer_id,order_count


- PySpark

In [0]:
from pyspark.sql.functions import count, col

repeat_customers = sales_clean.groupBy("customer_id") \
    .agg(count("sale_id").alias("order_count")) \
    .filter(col("order_count") > 2) \
    .orderBy(col("order_count").desc())

display(repeat_customers)

customer_id,order_count


### Business Pipeline Exercise 4: Highest Spending Customer in Each City

In [0]:
customers_clean.createOrReplaceTempView("customers")
sales_clean.createOrReplaceTempView("sales")

In [0]:
%sql
WITH customer_spend AS (
    SELECT
        c.city,
        s.customer_id,
        SUM(s.total_amount) AS total_spend
    FROM customers c
    JOIN sales s
        ON c.customer_id = s.customer_id
    GROUP BY
        c.city,
        s.customer_id
),

ranked_customers AS (
    SELECT
        city,
        customer_id,
        total_spend,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS rn
    FROM customer_spend
)

SELECT
    city,
    customer_id,
    total_spend
FROM ranked_customers
WHERE rn = 1;

city,customer_id,total_spend
Albany,9,79.96
Atlanta,16,60.0
Baltimore,36,18.75
Boise,7,110.95
Boston,22,119.96
Centerville,2,114.97
Charlotte,37,39.98
Chicago,43,40.0
Columbus,30,71.97
Denver,40,40.0


In [0]:
from pyspark.sql.functions import sum, row_number, desc
from pyspark.sql.window import Window

# Join customers and sales
customer_sales = customers_clean.join(
    sales_clean,
    on="customer_id",
    how="inner"
)

# Calculate total spend per customer in each city
customer_spend = customer_sales.groupBy(
    "city",
    "customer_id"
).agg(
    sum("total_amount").alias("total_spend")
)

# Rank customers within each city
window_spec = Window.partitionBy("city").orderBy(desc("total_spend"))

highest_spender = customer_spend.withColumn(
    "rank",
    row_number().over(window_spec)
).filter("rank = 1")

display(highest_spender)

city,customer_id,total_spend,rank
Albany,9,79.96,1
Atlanta,16,60.0,1
Baltimore,36,18.75,1
Boise,7,110.95,1
Boston,22,119.96,1
Centerville,2,114.97,1
Charlotte,37,39.98,1
Chicago,43,40.0,1
Columbus,30,71.97,1
Denver,40,40.0,1


### Business Pipeline Exercise 5: Final Reporting Table
Goal: Build a report with:
- Customer ID
- City
- Total Spend
- Number of Sales

In [0]:
%sql
SELECT
    s.customer_id,
    c.city,
    SUM(s.total_amount) AS total_spend,
    COUNT(s.sale_id) AS sale_count
FROM customers c
JOIN sales s
ON c.customer_id = s.customer_id
GROUP BY c.city, s.customer_id
ORDER BY total_spend DESC;

customer_id,city,total_spend,sale_count
4,Riverside,135.45999999999998,2
22,Boston,119.96,1
2,Centerville,114.97,2
7,Boise,110.95,1
45,New York,109.96,1
24,St. Louis,99.95,1
15,Seattle,92.0,1
34,Washington,92.0,1
20,Detroit,89.97,1
39,Memphis,89.96,1


In [0]:
from pyspark.sql.functions import sum, count

final_report = customer_sales.groupBy(
    "city",
    "customer_id"
).agg(
    sum("total_amount").alias("total_spend"),
    count("sale_id").alias("sale_count")
).orderBy("total_spend", ascending=False)

display(final_report)

city,customer_id,total_spend,sale_count
Riverside,4,135.45999999999998,2
Boston,22,119.96,1
Centerville,2,114.97,2
Boise,7,110.95,1
New York,45,109.96,1
St. Louis,24,99.95,1
Seattle,15,92.0,1
Washington,34,92.0,1
Detroit,20,89.97,1
Memphis,39,89.96,1


### Final Challenge: Reusable ETL Pipeline

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Phase3_ETL").getOrCreate()

# Extract
def read_data(path):
    return spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(path)

# Transform
def clean_data(df):
    return df.dropna().dropDuplicates()

# Load
def load_data(df):
    display(df)

# Read data
customers = read_data("/Volumes/workspace/default/databricks2027/customers.csv")
sales = read_data("/Volumes/workspace/default/databricks2027/sales.csv")

# Clean data
customers_clean = clean_data(customers)
sales_clean = clean_data(sales)

# Join datasets
customer_sales = customers_clean.join(
    sales_clean,
    on="customer_id",
    how="inner"
)

# Final report
report = customer_sales.groupBy(
    "city",
    "customer_id"
).agg(
    sum("total_amount").alias("total_spend"),
    count("sale_id").alias("sale_count")
)

load_data(report)

city,customer_id,total_spend,sale_count
Riverside,4,135.45999999999998,2
Charlotte,19,29.99,1
Boston,22,119.96,1
Chicago,43,40.0,1
Springfield,1,69.97,2
Milwaukee,26,66.75,1
Centerville,2,114.97,2
Denver,13,34.0,1
Indianapolis,18,59.97,2
Jacksonville,21,49.98,1
